In [ ]:
import numpy as np
import networkx as nx
from sklearn.neighbors import NearestNeighbors
import pickle

In [ ]:
def build_mood_graph(df: pd.DataFrame, k: int = 10) -> nx.DiGraph:
    """
    Build a k-NN graph in valence-energy space.
    Each track connects to its k nearest mood neighbours.
    Edge weight = Euclidean distance in mood space (lower = easier transition).
    """
    coords = df[["valence", "energy"]].values

    knn = NearestNeighbors(n_neighbors=k+1, metric="euclidean")
    knn.fit(coords)
    distances, indices = knn.kneighbors(coords)

    G = nx.DiGraph()

    # Add all nodes with attributes
    for i, row in df.iterrows():
        G.add_node(row["id"], valence=row["valence"], energy=row["energy"],
                   name=row["name"], artist=row["artist"],
                   preview_url=row["preview_url"], album_image=row["album_image"])

    # Add edges (skip self-loop at index 0)
    for i in range(len(df)):
        src_id = df.iloc[i]["id"]
        for j, dist in zip(indices[i][1:], distances[i][1:]):
            tgt_id = df.iloc[j]["id"]
            G.add_edge(src_id, tgt_id, weight=float(dist))

    return G

df = pd.read_parquet("data/tracks.parquet")
G = build_mood_graph(df, k=10)

with open("models/mood_graph.pkl", "wb") as f:
    pickle.dump(G, f)

print(f"Graph: {G.number_of_nodes()} nodes, {G.number_of_edges()} edges")